In [ ]:
!mkdir -p ~/.kaggle
!cp "kaggle (2).json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets list

ref                                                               title                                              size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  -------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
debayank2024/netflix-movies-and-series                            Netflix Movies and Series                       1400865  2026-07-27 09:40:46.527000           5907        137                1  
harshadapatil31/student-performance-and-study-habits-dataset      Student Performance & Study Habits Dataset        15035  2026-07-06 06:28:52.743000          12217        229                1  
srisyra02/spotify-music-artist-streaming-analytics                Spotify Music Artist Streaming Analytics          23463  2026-08-07 03:20:03.313000           1996         31                1  
jatinkhandelwal112/indian

In [ ]:
!kaggle datasets download -d puneet6060/intel-image-classification

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
100% 346M/346M [00:01<00:00, 196MB/s]



In [ ]:
import zipfile
unzip = zipfile.ZipFile('/content/intel-image-classification.zip')
unzip.extractall()
unzip.close()

In [ ]:
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import os
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom

In [ ]:
train = keras.utils.image_dataset_from_directory(
    '/content/seg_train/seg_train',
    validation_split=0.2,
    subset='training',
    seed=42,
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

validation = keras.utils.image_dataset_from_directory(
    '/content/seg_train/seg_train',
    validation_split=0.2,
    subset='validation',
    seed=42,
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

test = keras.utils.image_dataset_from_directory(
    '/content/seg_test/seg_test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256),
    shuffle=False
)

pred = keras.utils.image_dataset_from_directory(
    '/content/seg_pred/seg_pred',
    labels=None,
    batch_size=32,
    image_size=(256,256),
    shuffle=False
)

class_names = train.class_names
print(class_names)

Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Found 3000 files belonging to 6 classes.
Found 7301 files.
['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [ ]:
train_norm= train.map(lambda image, label: (image/255, label))
test_norm= test.map(lambda image, label: (image/255, label))
validation_norm = validation.map(lambda image, label: (image / 255, label))

In [ ]:
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1)
])

In [ ]:
model=Sequential()
]]
model.add(Conv2D(32,kernel_size=(3,3),padding='valid', activation='relu', input_shape=(256,256,3)))
model.add(MaxPool2D(pool_size=(2,2),strides=2, padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid', activation='relu'))
model.add(MaxPool2D(pool_size=(2,2),strides=2, padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid', activation='relu'))
model.add(MaxPool2D(pool_size=(2,2),strides=2, padding='valid'))

model.add(Conv2D(256,kernel_size=(3,3),padding='valid', activation='relu'))
model.add(MaxPool2D(pool_size=(2,2),strides=2, padding='valid'))

model.add(Flatten())

model.add(Dense(units=128, activation='relu'))
model.add(Dense(units=64, activation='relu'))
model.add(Dense(units=6, activation='softmax'))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │     6,422,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,819,718 (26.02 MB)

 Trainable params: 6,819,718 (26.02 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


In [ ]:
history = model.fit(
    train_norm,
    validation_data=validation_norm,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
351/351 ━━━━━━━━━━━━━━━━━━━━ 2533s 7s/step - accuracy: 0.5491 - loss: 1.1313 - val_accuracy: 0.6212 - val_loss: 0.9123
Epoch 2/20
351/351 ━━━━━━━━━━━━━━━━━━━━ 2550s 7s/step - accuracy: 0.6356 - loss: 0.9358 - val_accuracy: 0.6803 - val_loss: 0.8318
Epoch 3/20
351/351 ━━━━━━━━━━━━━━━━━━━━ 2340s 7s/step - accuracy: 0.6884 - loss: 0.8132 - val_accuracy: 0.7299 - val_loss: 0.7024
Epoch 4/20
262/351 ━━━━━━━━━━━━━━━━━━━━ 8:57 6s/step - accuracy: 0.7279 - loss: 0.7136

In [ ]:
test_loss, test_accuracy = model.evaluate(test_norm)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

NameError: name 'model' is not defined

In [ ]:
y_true = []
y_pred = []

for images, labels in test_norm:
    predictions = model.predict(images, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)


              precision    recall  f1-score   support

   buildings       0.75      0.81      0.78       437
      forest       0.90      0.97      0.93       474
     glacier       0.86      0.68      0.76       553
    mountain       0.68      0.84      0.75       525
         sea       0.84      0.81      0.82       510
      street       0.86      0.77      0.81       501

    accuracy                           0.81      3000
   macro avg       0.82      0.81      0.81      3000
weighted avg       0.82      0.81      0.81      3000



In [ ]:
predictions = model.predict(pred_norm)
predicted_numbers = np.argmax(predictions, axis=1)
predicted_names = [
    class_names[i]
    for i in predicted_numbers
]


file_names = [
    os.path.basename(path)
    for path in pred.file_paths
]

results = pd.DataFrame({
    'Image': file_names,
    'Predicted_Class': predicted_names,
    'Confidence': confidence
})

results.head(20)

229/229 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step


,Image,Predicted_Class,Confidence
0,10004.jpg,sea,98.166481
1,10005.jpg,street,99.945923
2,10012.jpg,buildings,90.600517
3,10013.jpg,buildings,89.917976
4,10017.jpg,mountain,83.602211
5,10021.jpg,buildings,97.341278
6,1003.jpg,buildings,97.214386
7,10034.jpg,glacier,99.999977
8,10038.jpg,buildings,94.372719
9,10040.jpg,forest,93.625175
